In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

ROOT = Path("..")

PROCESSED_PATH = (
    ROOT
    / "data"
    / "processed"
)

MODEL_PATH = (
    ROOT
    / "models"
)

REPORT_PATH = (
    ROOT
    / "reports"
)

In [2]:
train = pd.read_csv(
    PROCESSED_PATH / "D:\\My_Project\\Analyse\\capston Project_2-Global Supply Chain Risk & Logistics\\data\\processed\\train.csv"
)

validation = pd.read_csv(
    PROCESSED_PATH / "D:\\My_Project\\Analyse\\capston Project_2-Global Supply Chain Risk & Logistics\\data\\processed\\validation.csv"
)

comparison = pd.read_csv(
    REPORT_PATH / "D:\\My_Project\\Analyse\\capston Project_2-Global Supply Chain Risk & Logistics\\reports\\model_comparison.csv"
)

best_model_name = (
    comparison
    .sort_values(
        by=[
            "F1",
            "ROC_AUC",
            "Recall"
        ],
        ascending=False
    )
    .iloc[0]["Model"]
)

print(
    "Best model:",
    best_model_name
)

Best model: XGBoost


In [3]:
combined = pd.concat(
    [
        train,
        validation
    ],
    ignore_index=True
)

X = combined.drop(
    columns=["disruption"]
)

y = combined["disruption"].astype(int)

print(
    "Final training shape:",
    X.shape
)

Final training shape: (4000, 12)


In [4]:
numeric_features = (
    X
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [5]:
if best_model_name == "Random Forest":

    final_model = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

else:

    final_model = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

In [6]:
best_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            final_preprocessor
        ),
        (
            "model",
            final_model
        )
    ]
)

best_pipeline.fit(
    X,
    y
)

print(
    "Best model retrained successfully."
)

Best model retrained successfully.


In [7]:
BEST_MODEL_PATH = (
    MODEL_PATH
    / "best_model.pkl"
)

joblib.dump(
    best_pipeline,
    BEST_MODEL_PATH
)

print(
    "Best model saved:"
)

print(
    BEST_MODEL_PATH.resolve()
)

Best model saved:
D:\My_Project\Analyse\capston Project_2-Global Supply Chain Risk & Logistics\models\best_model.pkl
